# Soundstream-audio-codec

Download repository and install requirements

In [ ]:
!git clone "https://github.com/drAlexAK/soundstream-audio-codec-reimpl.git" ./soundstream
%cd soundstream
!pip install -q -r requirements.txt

In [ ]:
import torch
import torchaudio
import requests
from pathlib import Path
from IPython.display import display, Audio
from hydra.utils import instantiate
from omegaconf import OmegaConf

Download weights

In [ ]:
!bash src/scripts/download_model.sh

Load the model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

config = OmegaConf.load("pretrained/configs/model.yaml")
model = instantiate(config.model).to(device)

checkpoint = torch.load("pretrained/checkpoints/best.pth", map_location=device, weights_only=False)
state_dict = checkpoint["state_dict"]
model.load_state_dict(state_dict)
model.eval()

sample_rate = config.sample_rate

#### Note that model is trained for 16kHz audio files only

Provide an audio file to process

In [ ]:
url = "https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav"

Download file

In [ ]:
audio_path = Path("audio.wav")

response = requests.get(url)
response.raise_for_status()
audio_path.write_bytes(response.content)

audio, sr = torchaudio.load(str(audio_path))
print(audio.shape, sr)

Process file

In [ ]:
if sr != sample_rate:
      audio = torchaudio.functional.resample(audio, sr, sample_rate)

x = audio.unsqueeze(1).to(device)

with torch.no_grad():
      encoded = model.encoder(x)
      quantized, codes = model.quantizer(encoded)
      reconstructed = model.decoder(quantized)

reconstructed = reconstructed[..., : x.shape[-1]].squeeze(1).cpu()

print("encoded:", encoded.shape)
print("quantized:", quantized.shape)
print("codes:", codes.shape)

display(Audio(audio.numpy(), rate=sample_rate))
display(Audio(reconstructed.numpy(), rate=sample_rate))